# 500 · Concurrency & parallelism — procedural layer

**Mnemonic — FIVE dining philosophers around the table:** five forks, five thinkers, and every classic failure of shared state at one dinner.

Codes in this notebook: **500, 514, 520, 532, 541, 551, 565, 570, 584, 599**

**Method:** for every code cell — read it, **PREDICT** the exact output (values *and* order and rough timings), run it, **COMPARE**. A wrong prediction is the lesson: reread until the output feels inevitable. Run cells top to bottom in a single kernel.

No cell in this notebook is supposed to raise, and nothing here can hang: every `join`, `acquire`, and `wait` carries a timeout, demos are stdlib-only, and randomness is seeded (`random.seed(42)`) so misses and retries are reproducible.

## 500 · Concurrency vs parallelism

Concurrency structures a program to handle many tasks at once; parallelism actually executes tasks simultaneously on multiple cores.

*One barista juggling five orders by switching between them, versus five baristas each pulling an espresso shot at the same instant.*

**Watch:** four 0.2s tasks back to back vs overlapped in threads. Predict both elapsed times — then ask whether the threaded version would still win if the tasks were pure computation instead of sleeps.

In [1]:
import time, threading

def order(i):
    time.sleep(0.2)          # stands in for I/O: network call, disk read...

t0 = time.perf_counter()
for i in range(4):           # one barista, orders back to back
    order(i)
sequential = time.perf_counter() - t0

t0 = time.perf_counter()
threads = [threading.Thread(target=order, args=(i,), daemon=True) for i in range(4)]
for t in threads:
    t.start()                # all four sleeps overlap
for t in threads:
    t.join(timeout=5)
threaded = time.perf_counter() - t0

print(f"sequential: {sequential:.2f}s   (4 x 0.2s, one after another)")
print(f"threaded:   {threaded:.2f}s   (4 x 0.2s, overlapping)")
print("note: sleep releases the GIL, so this win is real for I/O-bound work only -")
print("for pure CPU work the GIL lets one thread run Python bytecode at a time.")

sequential: 0.80s   (4 x 0.2s, one after another)
threaded:   0.20s   (4 x 0.2s, overlapping)
note: sleep releases the GIL, so this win is real for I/O-bound work only -
for pure CPU work the GIL lets one thread run Python bytecode at a time.


## 514 · Non-atomic increment

`x += 1` is really read, add, write — three machine steps another thread can interleave between, silently losing counts.

*Updating a scoreboard by carrying the number card to a back desk, penciling +1, and carrying it back — meanwhile someone swapped the board.*

**Watch:** how far below 10000 the final count lands. The `time.sleep(0)` between read and write forces the OS to interleave the threads *every* iteration, so the lost updates that normally strike "sometimes, at 3am" happen right now, on demand.

In [2]:
import threading, time

counter = [0]
N = 5000

def worker():
    for _ in range(N):
        tmp = counter[0]          # 1. READ the shared value
        time.sleep(0)             # yield - the other thread sneaks in HERE
        counter[0] = tmp + 1      # 3. WRITE, clobbering whatever it did

threads = [threading.Thread(target=worker, daemon=True) for _ in range(2)]
for t in threads:
    t.start()
for t in threads:
    t.join(timeout=5)

print(f"expected: {2 * N}")
print(f"actual:   {counter[0]}")
print(f"lost updates: {2 * N - counter[0]} increments overwritten mid-flight")

expected: 10000
actual:   5000
lost updates: 5000 increments overwritten mid-flight


## 520 · Mutex

A mutual-exclusion lock with an owner: the thread that locks it must be the one to unlock it, admitting one holder at a time.

*A restroom key chained to your own belt loop: you carry it in, and only you can hang it back — nobody else may return your key.*

**Watch:** the identical racy loop from 514 runs twice — bare, then wrapped in a `threading.Lock`. Predict both numbers before running: the `sleep(0)` hazard is still there, but inside the lock nobody can interleave the read-modify-write.

In [3]:
import threading, time

N = 5000

def run(with_lock):
    counter = [0]
    lock = threading.Lock()
    def worker():
        for _ in range(N):
            if with_lock and not lock.acquire(timeout=5):
                return                     # safety valve, never taken here
            tmp = counter[0]
            time.sleep(0)                  # same hazard as in 514
            counter[0] = tmp + 1
            if with_lock:
                lock.release()             # only the holder releases its key
    threads = [threading.Thread(target=worker, daemon=True) for _ in range(2)]
    for t in threads:
        t.start()
    for t in threads:
        t.join(timeout=5)
    return counter[0]

racy = run(with_lock=False)
locked = run(with_lock=True)
print(f"expected: {2 * N} | racy: {racy} | with mutex: {locked}")
print("the mutex makes read-modify-write one indivisible step - exact count, every run")

expected: 10000 | racy: 5001 | with mutex: 10000
the mutex makes read-modify-write one indivisible step - exact count, every run


## 532 · Lock ordering

Impose a global order on locks and always acquire in ascending order — circular wait becomes structurally impossible.

*Mountaineers who must clip carabiners strictly in the order 1, 2, 3 up the cliff face — no rope can ever loop back into a circle.*

**Watch:** first cell — two philosophers grab opposite forks, then each times out reaching for the other's (`acquire(timeout=0.5)` is the only reason this prints instead of freezing). Second cell — the *same* two threads, but both take A before B: the cycle cannot form.

In [4]:
import threading

lock_a, lock_b = threading.Lock(), threading.Lock()
both_armed = threading.Barrier(2)      # everyone holds a fork before reaching over
both_done = threading.Barrier(2)       # nobody puts a fork down until both gave up
got = {}

def philosopher(first, second, key):
    with first:                        # grab own fork (t1: A first, t2: B first!)
        try:
            both_armed.wait(timeout=2)
            got[key] = second.acquire(timeout=0.5)   # reach for the other's fork
            if got[key]:
                second.release()
            both_done.wait(timeout=2)
        except threading.BrokenBarrierError:
            pass                       # safety valve only; never taken here

t1 = threading.Thread(target=philosopher, args=(lock_a, lock_b, "t1 wants B"), daemon=True)
t2 = threading.Thread(target=philosopher, args=(lock_b, lock_a, "t2 wants A"), daemon=True)
t1.start(); t2.start()
t1.join(timeout=5); t2.join(timeout=5)

print(f"t1 got B: {got.get('t1 wants B')}   t2 got A: {got.get('t2 wants A')}")
if not got.get("t1 wants B") and not got.get("t2 wants A"):
    print("potential deadlock detected: each holds one lock and waits on the other (circular wait)")
    print("only the acquire timeouts keep this cell from hanging forever")

t1 got B: False   t2 got A: False
potential deadlock detected: each holds one lock and waits on the other (circular wait)
only the acquire timeouts keep this cell from hanging forever


In [5]:
import threading, time

lock_a, lock_b = threading.Lock(), threading.Lock()
log = []

def philosopher(name):
    if lock_a.acquire(timeout=2):                  # EVERYONE reaches for A first...
        time.sleep(0.05)
        if lock_b.acquire(timeout=2):              # ...then B - no cycle can form
            log.append(f"{name}: got A then B, ate, released both")
            lock_b.release()
        lock_a.release()

threads = [threading.Thread(target=philosopher, args=(f"t{i}",), daemon=True) for i in (1, 2)]
for t in threads:
    t.start()
for t in threads:
    t.join(timeout=5)

for line in log:
    print(line)
print("same locks, same threads - one global acquisition order removes deadlock by construction")

t1: got A then B, ate, released both
t2: got A then B, ate, released both
same locks, same threads - one global acquisition order removes deadlock by construction


## 541 · Compare-and-swap (CAS)

A CPU instruction that atomically writes a new value only if the location still holds the expected old value, reporting success or failure.

*A vending machine that swaps in your new bottle only if the slot still shows exactly the barcode you claimed was there — else it spits your coin back.*

**Watch:** the retry count. Each failed CAS means an interloper changed the cell between our read and our write, so we re-read and retry instead of clobbering. Predict the relationship between `final`, our 20 increments, and `retries`.

In [6]:
import random

random.seed(42)

def compare_and_swap(cell, expected, new):
    # pretend this body is ONE atomic CPU instruction (CMPXCHG on x86)
    if cell[0] == expected:
        cell[0] = new
        return True
    return False

cell = [0]
retries = 0

for _ in range(20):                    # we want to add exactly 20
    while True:
        old = cell[0]                  # 1. read
        if random.random() < 0.3:      # simulated contention: an interloper
            cell[0] = old + 1          #    bumps the cell between our read and write
        if compare_and_swap(cell, old, old + 1):
            break                      # 2. our write landed on an unchanged cell
        retries += 1                   # 3. cell moved under us: loop, re-read, retry

print(f"final value:   {cell[0]}")
print(f"retries taken: {retries}")
print(f"final == our 20 + interlopers' bumps == 20 + {retries}: {cell[0] == 20 + retries}")
print("no lock, no lost update: failure is detected and answered with a retry")

final value:   33
retries taken: 13
final == our 20 + interlopers' bumps == 20 + 13: True
no lock, no lost update: failure is detected and answered with a retry


## 551 · Channel

A typed conduit through which concurrent tasks send and receive values, synchronizing as the data flows.

*A pneumatic bank tube between tellers: capsules whoosh across the ceiling, and the tube itself paces who waits for whom.*

**Watch:** items arrive in send order, and the `None` sentinel plays the role of `close()`. The `maxsize=2` bound is the point — a full tube makes the producer wait, so the channel *synchronizes*, it doesn't just store.

In [7]:
import threading, queue

ch = queue.Queue(maxsize=2)            # bounded: a full channel paces the producer

def producer():
    for capsule in ["espresso", "latte", "flat white", "mocha"]:
        ch.put(capsule, timeout=2)     # blocks (bounded) when the tube is full
    ch.put(None, timeout=2)            # sentinel: "channel closed"

t = threading.Thread(target=producer, daemon=True)
t.start()

consumed = []
while True:
    capsule = ch.get(timeout=2)        # blocks (bounded) when the tube is empty
    if capsule is None:
        break
    consumed.append(capsule)

t.join(timeout=5)
print("consumed in order:", consumed)
print("queue.Queue + sentinel = a channel: data flows one way, synchronization flows both")

consumed in order: ['espresso', 'latte', 'flat white', 'mocha']
queue.Queue + sentinel = a channel: data flows one way, synchronization flows both


## 565 · Async/await

Syntax that suspends a coroutine at `await` and resumes it when the awaited future resolves — async code shaped like sync code.

*A bookmark dropped mid-sentence when the doorbell rings: the reader handles the door, returns, and resumes on the exact same syllable.*

**Watch:** two 0.3s awaits gathered on a **single thread** finish in ~0.3s, not 0.6s. Then in the second cell, predict the exact print order — both coroutines start before either resumes. (The notebook kernel already runs an event loop, so we use top-level `await`; calling `asyncio.run` here would fail.)

In [8]:
import asyncio, time

t0 = time.perf_counter()
await asyncio.gather(asyncio.sleep(0.3), asyncio.sleep(0.3))
elapsed = time.perf_counter() - t0
print(f"two 0.3s awaits, gathered: {elapsed:.2f}s - overlapped on ONE thread, not 0.6s")

two 0.3s awaits, gathered: 0.30s - overlapped on ONE thread, not 0.6s


In [9]:
import asyncio

async def brew(name, delay):
    print(f"{name}: start")
    await asyncio.sleep(delay)         # bookmark dropped: the loop runs the other coroutine
    print(f"{name}: resume after {delay}s")

await asyncio.gather(brew("slow", 0.2), brew("fast", 0.1))
print("both started before either resumed - await marks exactly where each one paused")

slow: start
fast: start
fast: resume after 0.1s


slow: resume after 0.2s
both started before either resumed - await marks exactly where each one paused


## 570 · Thread pool

A fixed crew of reusable worker threads pulling tasks from a queue — amortizes thread creation and caps total concurrency.

*A hotel keeps exactly five lifeguards rotating at the pool: swimmers (tasks) queue for attention; nobody hires a fresh lifeguard per swimmer.*

**Watch:** 8 tasks of 0.2s through 4 workers = two waves ≈ 0.4s, versus 1.6s serial. Predict both timings and whether the result lists match.

In [10]:
import time
from concurrent.futures import ThreadPoolExecutor

def swim(i):
    time.sleep(0.2)                    # each swimmer needs 0.2s of attention
    return i * i

t0 = time.perf_counter()
serial = [swim(i) for i in range(8)]
t_serial = time.perf_counter() - t0

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:   # exactly 4 lifeguards
    pooled = list(pool.map(swim, range(8)))       # 8 swimmers queue up
t_pool = time.perf_counter() - t0

print(f"serial, 8 x 0.2s:      {t_serial:.2f}s")
print(f"pool of 4 workers:     {t_pool:.2f}s   (8 tasks / 4 workers = 2 waves of 0.2s)")
print("results identical:", serial == pooled, "- map keeps submission order")

serial, 8 x 0.2s:      1.60s
pool of 4 workers:     0.40s   (8 tasks / 4 workers = 2 waves of 0.2s)
results identical: True - map keeps submission order


## 584 · Map-reduce pattern

Map a function over records in parallel, then reduce the intermediate results by key into aggregates — with a shuffle in between.

*A national census: thousands of counters tally their own street (map), mail totals to per-state desks (shuffle), which sum them into one book (reduce).*

**Watch:** four independent partial `Counter`s (map) merge key-by-key into one (reduce). Counter addition is associative, so the arrival order of partials cannot change the answer — that indifference is what makes the pattern parallelizable.

In [11]:
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from functools import reduce

text = ("the philosophers eat rice while the philosophers think "
        "and the forks wait while the rice cools ") * 4
words = text.split()
chunks = [words[i::4] for i in range(4)]           # split the census into 4 streets

def map_count(chunk):                              # MAP: each worker tallies its own chunk
    return Counter(chunk)

with ThreadPoolExecutor(max_workers=4) as pool:
    partials = list(pool.map(map_count, chunks))

total = reduce(lambda a, b: a + b, partials)       # REDUCE: merge tallies key by key

print("words processed:", len(words), "in", len(partials), "partial counters")
print("top words:", total.most_common(3))
print("full counts:", dict(sorted(total.items())))

words processed: 64 in 4 partial counters
top words: [('the', 16), ('while', 8), ('philosophers', 8)]
full counts: {'and': 4, 'cools': 4, 'eat': 4, 'forks': 4, 'philosophers': 8, 'rice': 8, 'the': 16, 'think': 4, 'wait': 4, 'while': 8}


## 599 · Never sync by sleeping

`sleep(n)` is a timing guess, not synchronization; wait on the actual condition with joins, events, barriers, or queues.

*Leaving the package on the porch and napping until the courier has 'probably' come — versus a doorbell that rings at the actual pickup.*

**Watch:** the producer takes a seeded ~0.21s. The consumer that naps a fixed 0.1s then reads sees `None` — its guess lost the race. The `Event.wait(timeout=2)` consumer always sees the value: its timeout is a safety bound on a *real condition*, not a guess about timing.

In [12]:
import threading, random, time

random.seed(42)
delay = random.uniform(0.05, 0.3)          # seeded: reproducible, but a guess can't know it
print(f"(this run the courier takes {delay:.3f}s)")

# BAD - sleep and hope --------------------------------------------------
porch = {"package": None}
def courier_bad():
    time.sleep(delay)
    porch["package"] = "delivered"

t1 = threading.Thread(target=courier_bad, daemon=True)
t1.start()
time.sleep(0.1)                            # 'a tenth of a second is probably enough'
print("sleep-then-read saw:", porch["package"], " <- None: the nap ended before the courier came")
t1.join(timeout=5)

# GOOD - wait on the actual condition -----------------------------------
porch2 = {"package": None}
doorbell = threading.Event()
def courier_good():
    time.sleep(delay)
    porch2["package"] = "delivered"
    doorbell.set()                         # ring when it actually happened

t2 = threading.Thread(target=courier_good, daemon=True)
t2.start()
rang = doorbell.wait(timeout=2)            # bounded wait on the REAL event
print("event-wait saw:     ", porch2["package"], f" (doorbell rang: {rang})")
t2.join(timeout=5)

print("sleep guesses at time; the Event waits on the fact - and its timeout is a")
print("safety bound for the pathological case, never the synchronization itself")

(this run the courier takes 0.210s)
sleep-then-read saw: None  <- None: the nap ended before the courier came


event-wait saw:      delivered  (doorbell rang: True)
sleep guesses at time; the Event waits on the fact - and its timeout is a
safety bound for the pathological case, never the synchronization itself
